In [1]:
!pip install -q torch transformers pandas scikit-learn sentencepiece datasets

In [2]:
!unzip /content/SubTaskA.zip

Archive:  /content/SubTaskA.zip
   creating: SubTaskA/
   creating: SubTaskA/SubTaskA/
   creating: SubTaskA/SubTaskA/Data/
  inflating: SubTaskA/SubTaskA/Data/dev.csv  
  inflating: SubTaskA/SubTaskA/Data/dev_gold.csv  
  inflating: SubTaskA/SubTaskA/Data/dev_submission_format.csv  
  inflating: SubTaskA/SubTaskA/Data/eval.csv  
  inflating: SubTaskA/SubTaskA/Data/eval_submission_format.csv  
  inflating: SubTaskA/SubTaskA/Data/train_one_shot.csv  
  inflating: SubTaskA/SubTaskA/Data/train_zero_shot.csv  
   creating: SubTaskA/SubTaskA/TestData/
  inflating: SubTaskA/SubTaskA/TestData/test.csv  
  inflating: SubTaskA/SubTaskA/TestData/test_submission_format.csv  
  inflating: SubTaskA/SubTaskA/TestData/train_one_shot.csv  


In [4]:
import pandas as pd
from pathlib import Path
import os

# Paths (relative to /content in Colab)
INPUT_DIR = Path("/content/SubTaskA/SubTaskA/Data/")  # Upload your Data folder here
OUTPUT_DIR = Path("Data/ZeroShot")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Helper: concatenate context
def build_sentence(row):
    prev = str(row['Previous']).strip() if pd.notna(row['Previous']) else ""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    nxt = str(row['Next']).strip() if pd.notna(row['Next']) else ""
    return f"{prev} {target} {nxt}".strip()

# Load and prepare TRAIN data
print("Processing train data...")
train_df = pd.read_csv(INPUT_DIR / "train_zero_shot.csv")
train_out = pd.DataFrame({
    "label": train_df["Label"],
    "sentence1": train_df.apply(build_sentence, axis=1)
})
train_out.to_csv(OUTPUT_DIR / "train.csv", index=False)
print(f"  ✓ Saved train.csv ({len(train_out)} rows)")

# Load and prepare DEV data (merge with gold labels)
print("Processing dev data...")
dev_df = pd.read_csv(INPUT_DIR / "dev.csv")
dev_gold = pd.read_csv(INPUT_DIR / "dev_gold.csv")
dev_df = dev_df.merge(dev_gold[["ID", "Label"]], on="ID", how="left")
dev_out = pd.DataFrame({
    "label": dev_df["Label"],
    "sentence1": dev_df.apply(build_sentence, axis=1)
})
dev_out.to_csv(OUTPUT_DIR / "dev.csv", index=False)
print(f"  ✓ Saved dev.csv ({len(dev_out)} rows)")

# Load and prepare EVAL data (no gold labels yet)
print("Processing eval data...")
eval_df = pd.read_csv(INPUT_DIR / "eval.csv")
eval_out = pd.DataFrame({
    "label": 1,  # dummy label for submission format
    "sentence1": eval_df.apply(build_sentence, axis=1)
})
eval_out.to_csv(OUTPUT_DIR / "eval.csv", index=False)
print(f"  ✓ Saved eval.csv ({len(eval_out)} rows)")

print("\n✅ Data preparation complete!")

Processing train data...
  ✓ Saved train.csv (4491 rows)
Processing dev data...
  ✓ Saved dev.csv (739 rows)
Processing eval data...
  ✓ Saved eval.csv (762 rows)

✅ Data preparation complete!


In [5]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import f1_score

print("Loading data...")
train_df = pd.read_csv("Data/ZeroShot/train.csv")
dev_df = pd.read_csv("Data/ZeroShot/dev.csv")

# Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df[["label", "sentence1"]])
dev_ds = Dataset.from_pandas(dev_df[["label", "sentence1"]])

print(f"  ✓ Train: {len(train_ds)} samples")
print(f"  ✓ Dev: {len(dev_ds)} samples")

# Tokenize
print("\nTokenizing...")
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_length = 128

def tokenize_fn(batch):
    return tokenizer(
        batch["sentence1"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
dev_ds = dev_ds.map(tokenize_fn, batched=True)

# Keep only model inputs
cols_to_keep = ["input_ids", "attention_mask", "token_type_ids", "label"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in cols_to_keep])
dev_ds = dev_ds.remove_columns([c for c in dev_ds.column_names if c not in cols_to_keep])

print("  ✓ Tokenization complete")

# Load model
print("\nLoading model...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print("  ✓ Model loaded")

# Define metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds, average="macro")}

# Training arguments
training_args = TrainingArguments(
    output_dir="models/ZeroShot/0",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=9,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    seed=0,
    report_to="none",
)

# Trainer
print("\nStarting training...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
print("\n✅ Training complete!")

Loading data...
  ✓ Train: 4491 samples
  ✓ Dev: 739 samples

Tokenizing...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ✓ Model loaded

Starting training...


/tmp/ipython-input-179276271.py:75: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,No log,0.670335,0.633972
2,No log,0.759273,0.649084
3,No log,1.082914,0.675299
4,0.307900,1.434792,0.683124
5,0.307900,1.692758,0.689069
6,0.307900,2.043936,0.682969
7,0.307900,2.147163,0.694548
8,0.020300,2.183620,0.684478
9,0.020300,2.191588,0.692321



✅ Training complete!


In [7]:
import torch

print("Generating predictions on dev set...")

dev_df = pd.read_csv("Data/ZeroShot/dev.csv")
dev_encodings = tokenizer(
    dev_df["sentence1"].tolist(),
    truncation=True, padding=True, max_length=128, return_tensors="pt"
)

model.eval()
dev_preds = []
with torch.no_grad():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    input_ids = dev_encodings["input_ids"].to(device)
    attention_mask = dev_encodings["attention_mask"].to(device)

    for i in range(0, len(input_ids), 32):
        batch_input = input_ids[i:i+32]
        batch_attn = attention_mask[i:i+32]
        outputs = model(batch_input, attention_mask=batch_attn)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        dev_preds.extend(preds)

# Save predictions
import os
os.makedirs("models/ZeroShot/0/eval-dev", exist_ok=True)
pred_df = pd.DataFrame({
    "index": list(range(len(dev_preds))),
    "prediction": dev_preds
})
pred_df.to_csv("models/ZeroShot/0/eval-dev/test_results_None.txt", sep="\t", index=False)
print(f"✓ Saved predictions ({len(dev_preds)} samples)")

Generating predictions on dev set...
✓ Saved predictions (739 samples)


In [9]:
print("Computing Macro F1 score...")

# Load gold labels (in dev.csv order)
dev_input = pd.read_csv("/content/SubTaskA/SubTaskA/Data/dev.csv")
dev_gold = pd.read_csv("/content/SubTaskA/SubTaskA/Data/dev_gold.csv").set_index("ID")

# Map labels to dev order
gold_labels = dev_input["ID"].map(dev_gold["Label"]).values

# Load predictions
pred_file = pd.read_csv("models/ZeroShot/0/eval-dev/test_results_None.txt", sep="\t")
pred_labels = pred_file["prediction"].values

# Compute Macro F1
macro_f1 = f1_score(gold_labels, pred_labels, average="macro")

print(f"\n{'='*40}")
print(f"MACRO F1 SCORE: {macro_f1:.4f}")
print(f"{'='*40}")
print(f"\nGold labels shape: {gold_labels.shape}")
print(f"Predictions shape: {pred_labels.shape}")
print(f"Label 0 (idiom): {(gold_labels == 0).sum()}")
print(f"Label 1 (literal): {(gold_labels == 1).sum()}")
print(f"Predicted 0: {(pred_labels == 0).sum()}")
print(f"Predicted 1: {(pred_labels == 1).sum()}")

Computing Macro F1 score...

MACRO F1 SCORE: 0.6945

Gold labels shape: (739,)
Predictions shape: (739,)
Label 0 (idiom): 336
Label 1 (literal): 403
Predicted 0: 361
Predicted 1: 378


In [11]:
# Comprehensive F1 Score Analysis - Detailed Breakdown
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, classification_report
from pathlib import Path
import os

print("="*80)
print("COMPREHENSIVE F1 SCORE ANALYSIS - DETAILED BREAKDOWN")
print("="*80)

# Paths
INPUT_DIR = Path("/content/SubTaskA/SubTaskA/Data/") if os.path.exists("/content") else Path("Data")

# Load data with language information
dev_input = pd.read_csv(INPUT_DIR / "dev.csv")
dev_gold = pd.read_csv(INPUT_DIR / "dev_gold.csv")

# Merge to get language and labels
dev_full = dev_input.merge(dev_gold[["ID", "Label"]], on="ID", how="left")

# Load predictions
pred_file = pd.read_csv("models/ZeroShot/0/eval-dev/test_results_None.txt", sep="\t")

# Create comprehensive dataframe
results_df = pd.DataFrame({
    'ID': dev_full['ID'],
    'Language': dev_full['Language'],
    'Gold_Label': dev_full['Label'],
    'Predicted_Label': pred_file['prediction'].values
})

print(f"\nTotal samples: {len(results_df)}")
print(f"Languages: {results_df['Language'].unique()}")
print(f"Classes: 0 (Idiomatic), 1 (Literal)")

# ============================================================================
# TABLE 1: OVERALL RESULTS (ALL LANGUAGES COMBINED) - Using classification_report
# ============================================================================
print("\n" + "="*80)
print("TABLE 1: OVERALL RESULTS (ALL LANGUAGES COMBINED)")
print("="*80)

print("\n" + classification_report(
    results_df['Gold_Label'],
    results_df['Predicted_Label'],
    target_names=['Idiomatic', 'Literal'],
    digits=4
))

# Get macro F1 for later use
macro_f1 = f1_score(results_df['Gold_Label'], results_df['Predicted_Label'], average='macro')

# ============================================================================
# TABLE 2: RESULTS BY LANGUAGE - Using classification_report
# ============================================================================
print("\n\n" + "="*80)
print("TABLE 2: RESULTS BY LANGUAGE")
print("="*80)

for lang in sorted(results_df['Language'].unique()):
    lang_data = results_df[results_df['Language'] == lang]

    print(f"\n--- {lang} ({len(lang_data)} samples) ---")
    print(classification_report(
        lang_data['Gold_Label'],
        lang_data['Predicted_Label'],
        target_names=['Idiomatic', 'Literal'],
        digits=4
    ))

# ============================================================================
# SUMMARY - KEY METRICS
# ============================================================================
print("\n" + "="*80)
print("SUMMARY - KEY METRICS")
print("="*80)

# Calculate macro F1 for each language
en_data = results_df[results_df['Language'] == 'EN']
pt_data = results_df[results_df['Language'] == 'PT']

en_macro_f1 = f1_score(en_data['Gold_Label'], en_data['Predicted_Label'], average='macro')
pt_macro_f1 = f1_score(pt_data['Gold_Label'], pt_data['Predicted_Label'], average='macro')

summary_table = []
summary_table.append({'Metric': 'EN Macro F1', 'Value': f"{en_macro_f1:.4f}"})
summary_table.append({'Metric': 'PT Macro F1', 'Value': f"{pt_macro_f1:.4f}"})
summary_table.append({'Metric': 'Combined Macro F1', 'Value': f"{macro_f1:.4f}"})

summary_df = pd.DataFrame(summary_table)
print("\n" + summary_df.to_string(index=False))

print("\n" + "="*80)
print("✅ COMPREHENSIVE ANALYSIS COMPLETE!")
print("="*80)
print(f"""
KEY TAKEAWAYS:
- Overall Macro F1 (EN+PT): {macro_f1:.4f}
- EN Performance: {en_macro_f1:.4f}
- PT Performance: {pt_macro_f1:.4f}
""")

COMPREHENSIVE F1 SCORE ANALYSIS - DETAILED BREAKDOWN

Total samples: 739
Languages: ['EN' 'PT']
Classes: 0 (Idiomatic), 1 (Literal)

TABLE 1: OVERALL RESULTS (ALL LANGUAGES COMBINED)

              precision    recall  f1-score   support

   Idiomatic     0.6537    0.7024    0.6772       336
     Literal     0.7354    0.6898    0.7119       403

    accuracy                         0.6955       739
   macro avg     0.6946    0.6961    0.6945       739
weighted avg     0.6983    0.6955    0.6961       739



TABLE 2: RESULTS BY LANGUAGE

--- EN (466 samples) ---
              precision    recall  f1-score   support

   Idiomatic     0.6488    0.5989    0.6229       182
     Literal     0.7550    0.7923    0.7732       284

    accuracy                         0.7167       466
   macro avg     0.7019    0.6956    0.6980       466
weighted avg     0.7135    0.7167    0.7145       466


--- PT (273 samples) ---
              precision    recall  f1-score   support

   Idiomatic     0.6580 